In [0]:
%sql
-- 1. Create a dedicated catalog for your POC
CREATE CATALOG IF NOT EXISTS talk2db_poc;

-- 2. Create a schema for your "raw/silver" data
CREATE SCHEMA IF NOT EXISTS talk2db_poc.data_layer;

-- 3. Create a schema specifically for the AI's metadata (The "Brain")
CREATE SCHEMA IF NOT EXISTS talk2db_poc.ai_metadata;

In [0]:
%sql
CREATE TABLE talk2db_poc.data_layer.customer_metrics (
  customer_id INT,
  customer_name STRING,
  location STRING,
  status_code INT COMMENT 'Business Logic: 0 = Churned, 1 = Active, 2 = Prospect',
  total_spend DOUBLE,
  last_login_date DATE
) 
-- This makes it work with Vector Search later
TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Insert a few rows for testing
INSERT INTO talk2db_poc.data_layer.customer_metrics VALUES 
(101, 'Saketh', 'Charlotte', 1, 5000.00, '2026-03-20'),
(102, 'John Doe', 'Charlotte', 0, 1500.00, '2025-12-15'),
(103, 'Jane Smith', 'Fairfax', 0, 200.00, '2026-01-10');

COMMENT ON TABLE talk2db_poc.data_layer.customer_metrics IS 'Contains core customer profiles, spending habits, and status for churn analysis.';

In [0]:
%sql
CREATE TABLE talk2db_poc.ai_metadata.table_descriptions (
  table_name STRING,
  table_purpose STRING,
  business_rules STRING
);

INSERT INTO talk2db_poc.ai_metadata.table_descriptions VALUES 
('talk2db_poc.data_layer.customer_metrics', 
 'Used for finding customer details and location-based counts.', 
 'To find CHURNED users, filter where status_code = 0. To find ACTIVE users, status_code = 1.');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- Replace with your actual table name if different
ALTER TABLE talk2db_poc.ai_metadata.table_descriptions 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
index = client.get_index(endpoint_name="talk2db_poc_endpoint", index_name="talk2db_poc.ai_metadata.table_index")

# Simulate a user question
query = "How many customers in Charlotte have stopped using our service?"

results = index.similarity_search(
  query_text=query,
  columns=["table_name", "business_rules"],
  num_results=1
)

print(f"Top Table Found: {results['result']['data_array'][0][0]}")
print(f"Business Logic to use: {results['result']['data_array'][0][1]}")

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-8882013845911439>, line 1
----> 1 from databricks.vector_search.client import VectorSearchClient
      3 client = VectorSearchClient()
      4 index = client.get_index(endpoint_name="talk2db_poc_endpoint", index_name="talk2db_poc.ai_metadata.table_index")

ModuleNotFoundError: No module named 'databricks.vector_search'